In [1]:
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef
from imblearn.metrics import geometric_mean_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier

SCALED_DIR = Path("../Data/Processed")
RANDOM_SEED = 42

X_german_train = pd.read_csv(SCALED_DIR / "german_X_train.csv")
X_german_test = pd.read_csv(SCALED_DIR / "german_X_test.csv")
y_german_train = pd.read_csv(SCALED_DIR / "german_y_train.csv").squeeze()
y_german_test = pd.read_csv(SCALED_DIR / "german_y_test.csv").squeeze()

print(X_german_train.shape, X_german_test.shape)

(800, 40) (200, 40)


In [2]:
def evaluate_model(model, X_train, y_train, X_test, y_test, name):
    # Fits the given model/pipeline and returns the four standard metrics
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    return {
        "config": name,
        "AUC-ROC": roc_auc_score(y_test, y_proba),
        "F1 (minority)": f1_score(y_test, y_pred),
        "G-mean": geometric_mean_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

In [3]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# RF without SMOTE
rf_plain = RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200)
result_rf_plain = evaluate_model(rf_plain, X_german_train, y_german_train, X_german_test, y_german_test, "RF (no SMOTE)")

# RF with SMOTE - using imblearn Pipeline so SMOTE only applies to training folds
rf_smote = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200))
])
result_rf_smote = evaluate_model(rf_smote, X_german_train, y_german_train, X_german_test, y_german_test, "RF (with SMOTE)")

print(pd.DataFrame([result_rf_plain, result_rf_smote]))

            config   AUC-ROC  F1 (minority)    G-mean       MCC
0    RF (no SMOTE)  0.795655       0.525253  0.626973  0.393805
1  RF (with SMOTE)  0.784167       0.560748  0.662786  0.409160


In [5]:
import re

def clean_column_names(df):
    # XGBoost rejects feature names containing [, ], or < characters.
    # Replace these (and other potentially problematic characters) with underscores
    # to keep names safe across all models, not just XGBoost.
    df = df.copy()
    df.columns = [re.sub(r"[\[\]<>]", "_", str(col)) for col in df.columns]
    return df

X_german_train = clean_column_names(X_german_train)
X_german_test = clean_column_names(X_german_test)

print(X_german_train.columns.tolist())

['duration_months', 'credit_amount', 'savings_account', 'employment_since', 'installment_rate_pct', 'residence_since', 'age', 'existing_credits_count', 'job', 'num_dependents', 'status_checking_account__ 0 DM', 'status_checking_account__= 200 DM / salary assignment', 'status_checking_account_no checking account', 'credit_history_critical account/other credits existing', 'credit_history_delay in paying off in the past', 'credit_history_existing credits paid duly', 'credit_history_no credits/all paid duly', 'purpose_car (new)', 'purpose_car (used)', 'purpose_domestic appliances', 'purpose_education', 'purpose_furniture/equipment', 'purpose_others', 'purpose_radio/television', 'purpose_repairs', 'purpose_retraining', 'personal_status_sex_male: divorced/separated', 'personal_status_sex_male: married/widowed', 'personal_status_sex_male: single', 'other_debtors_guarantors_guarantor', 'other_debtors_guarantors_none', 'property_car or other', 'property_real estate', 'property_unknown/no proper

In [6]:
# XGBoost without SMOTE
xgb_plain = XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss")
result_xgb_plain = evaluate_model(xgb_plain, X_german_train, y_german_train, X_german_test, y_german_test, "XGBoost (no SMOTE)")

# XGBoost with SMOTE
xgb_smote = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
])
result_xgb_smote = evaluate_model(xgb_smote, X_german_train, y_german_train, X_german_test, y_german_test, "XGBoost (with SMOTE)")

print(pd.DataFrame([result_xgb_plain, result_xgb_smote]))

                 config   AUC-ROC  F1 (minority)    G-mean       MCC
0    XGBoost (no SMOTE)  0.778214       0.462963  0.590097  0.270803
1  XGBoost (with SMOTE)  0.794881       0.584071  0.686607  0.422756


In [7]:
# Stacked ensemble without SMOTE
stack_plain = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200)),
        ("xgb", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=skf
)
result_stack_plain = evaluate_model(stack_plain, X_german_train, y_german_train, X_german_test, y_german_test, "Stacked (no SMOTE)")

# Stacked ensemble with SMOTE
stack_smote = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200)),
            ("xgb", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=skf
    ))
])
result_stack_smote = evaluate_model(stack_smote, X_german_train, y_german_train, X_german_test, y_german_test, "Stacked (with SMOTE)")

print(pd.DataFrame([result_stack_plain, result_stack_smote]))

                 config   AUC-ROC  F1 (minority)    G-mean       MCC
0    Stacked (no SMOTE)  0.785833       0.500000  0.609938  0.354604
1  Stacked (with SMOTE)  0.789762       0.636364  0.721688  0.503953


In [8]:
german_results = pd.DataFrame([
    result_rf_plain, result_rf_smote,
    result_xgb_plain, result_xgb_smote,
    result_stack_plain, result_stack_smote
])
print(german_results)

                 config   AUC-ROC  F1 (minority)    G-mean       MCC
0         RF (no SMOTE)  0.795655       0.525253  0.626973  0.393805
1       RF (with SMOTE)  0.784167       0.560748  0.662786  0.409160
2    XGBoost (no SMOTE)  0.778214       0.462963  0.590097  0.270803
3  XGBoost (with SMOTE)  0.794881       0.584071  0.686607  0.422756
4    Stacked (no SMOTE)  0.785833       0.500000  0.609938  0.354604
5  Stacked (with SMOTE)  0.789762       0.636364  0.721688  0.503953


In [9]:
X_home_train = pd.read_csv(SCALED_DIR / "home_X_train.csv")
X_home_test = pd.read_csv(SCALED_DIR / "home_X_test.csv")
y_home_train = pd.read_csv(SCALED_DIR / "home_y_train.csv").squeeze()
y_home_test = pd.read_csv(SCALED_DIR / "home_y_test.csv").squeeze()

X_lending_train = pd.read_csv(SCALED_DIR / "lending_X_train.csv")
X_lending_test = pd.read_csv(SCALED_DIR / "lending_X_test.csv")
y_lending_train = pd.read_csv(SCALED_DIR / "lending_y_train.csv").squeeze()
y_lending_test = pd.read_csv(SCALED_DIR / "lending_y_test.csv").squeeze()

# Apply the same column-name cleaning fix proactively, since both datasets
# likely contain [, ], or < characters from one-hot encoded category labels
X_home_train = clean_column_names(X_home_train)
X_home_test = clean_column_names(X_home_test)
X_lending_train = clean_column_names(X_lending_train)
X_lending_test = clean_column_names(X_lending_test)

print(X_home_train.shape, X_lending_train.shape)

(246005, 100) (8000, 69)


In [10]:
#Home loan
# RF
rf_plain_home = RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200)
result_rf_plain_home = evaluate_model(rf_plain_home, X_home_train, y_home_train, X_home_test, y_home_test, "RF (no SMOTE)")

rf_smote_home = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200))
])
result_rf_smote_home = evaluate_model(rf_smote_home, X_home_train, y_home_train, X_home_test, y_home_test, "RF (with SMOTE)")

# XGBoost
xgb_plain_home = XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss")
result_xgb_plain_home = evaluate_model(xgb_plain_home, X_home_train, y_home_train, X_home_test, y_home_test, "XGBoost (no SMOTE)")

xgb_smote_home = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
])
result_xgb_smote_home = evaluate_model(xgb_smote_home, X_home_train, y_home_train, X_home_test, y_home_test, "XGBoost (with SMOTE)")

print(pd.DataFrame([result_rf_plain_home, result_rf_smote_home, result_xgb_plain_home, result_xgb_smote_home]))

                 config   AUC-ROC  F1 (minority)    G-mean       MCC
0         RF (no SMOTE)  0.723503       0.001610  0.028384  0.023807
1       RF (with SMOTE)  0.714293       0.015711  0.089688  0.039112
2    XGBoost (no SMOTE)  0.745622       0.059436  0.177571  0.109292
3  XGBoost (with SMOTE)  0.746414       0.062488  0.182055  0.116449


In [11]:
stack_plain_home = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200)),
        ("xgb", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=skf
)
result_stack_plain_home = evaluate_model(stack_plain_home, X_home_train, y_home_train, X_home_test, y_home_test, "Stacked (no SMOTE)")

stack_smote_home = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200)),
            ("xgb", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=skf
    ))
])
result_stack_smote_home = evaluate_model(stack_smote_home, X_home_train, y_home_train, X_home_test, y_home_test, "Stacked (with SMOTE)")

home_results = pd.DataFrame([
    result_rf_plain_home, result_rf_smote_home,
    result_xgb_plain_home, result_xgb_smote_home,
    result_stack_plain_home, result_stack_smote_home
])
print(home_results)

                 config   AUC-ROC  F1 (minority)    G-mean       MCC
0         RF (no SMOTE)  0.723503       0.001610  0.028384  0.023807
1       RF (with SMOTE)  0.714293       0.015711  0.089688  0.039112
2    XGBoost (no SMOTE)  0.745622       0.059436  0.177571  0.109292
3  XGBoost (with SMOTE)  0.746414       0.062488  0.182055  0.116449
4    Stacked (no SMOTE)  0.744065       0.086558  0.217534  0.132261
5  Stacked (with SMOTE)  0.721914       0.102408  0.245249  0.110690


In [12]:
#Lending club
# RF
rf_plain_lending = RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200)
result_rf_plain_lending = evaluate_model(rf_plain_lending, X_lending_train, y_lending_train, X_lending_test, y_lending_test, "RF (no SMOTE)")

rf_smote_lending = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200))
])
result_rf_smote_lending = evaluate_model(rf_smote_lending, X_lending_train, y_lending_train, X_lending_test, y_lending_test, "RF (with SMOTE)")

# XGBoost
xgb_plain_lending = XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss")
result_xgb_plain_lending = evaluate_model(xgb_plain_lending, X_lending_train, y_lending_train, X_lending_test, y_lending_test, "XGBoost (no SMOTE)")

xgb_smote_lending = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
])
result_xgb_smote_lending = evaluate_model(xgb_smote_lending, X_lending_train, y_lending_train, X_lending_test, y_lending_test, "XGBoost (with SMOTE)")

# Stacked ensemble
stack_plain_lending = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200)),
        ("xgb", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=skf
)
result_stack_plain_lending = evaluate_model(stack_plain_lending, X_lending_train, y_lending_train, X_lending_test, y_lending_test, "Stacked (no SMOTE)")

stack_smote_lending = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("clf", StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200)),
            ("xgb", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=skf
    ))
])
result_stack_smote_lending = evaluate_model(stack_smote_lending, X_lending_train, y_lending_train, X_lending_test, y_lending_test, "Stacked (with SMOTE)")

lending_results = pd.DataFrame([
    result_rf_plain_lending, result_rf_smote_lending,
    result_xgb_plain_lending, result_xgb_smote_lending,
    result_stack_plain_lending, result_stack_smote_lending
])
print(lending_results)

                 config   AUC-ROC  F1 (minority)    G-mean       MCC
0         RF (no SMOTE)  0.850970       0.285714  0.408248  0.405166
1       RF (with SMOTE)  0.816425       0.000000  0.000000  0.000000
2    XGBoost (no SMOTE)  0.907290       0.468085  0.552771  0.549286
3  XGBoost (with SMOTE)  0.884999       0.440000  0.552348  0.484822
4    Stacked (no SMOTE)  0.868169       0.468085  0.552771  0.549286
5  Stacked (with SMOTE)  0.826643       0.318182  0.440846  0.408497


In [13]:
# Check what RF is actually predicting after SMOTE - is it predicting
# zero positives entirely, or something else going on?
rf_smote_lending.fit(X_lending_train, y_lending_train)
y_pred_check = rf_smote_lending.predict(X_lending_test)
print(pd.Series(y_pred_check).value_counts())

# Also check SMOTE's k_neighbors default (5) against how few minority
# samples actually exist in training
print("Minority class count in training:", (y_lending_train == 1).sum())

0    2000
Name: count, dtype: int64
Minority class count in training: 142
